# Thread workers

> Threaded workers to isolate IO operations and CPU intensive tasks into a background thread

In [ ]:
#| default_exp threadworkers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import threading
import queue
import time

import logging


In [ ]:
#| exporti
syslog = logging.getLogger("root." + __name__)


In [ ]:
#| exporti

def create_put_task_item_fn(task_queue,  **put_kwargs):

    def put_task_item(task_queue, *work_args, **work_kwargs):
        task_queue.put(item=(work_args, work_kwargs), **put_kwargs)

    return put_task_item


In [ ]:
#| exporti
def create_get_task_item_fn(product_queue,  **get_kwargs):
    "Binds a queue and kwargs for getting items from the queue"

    def get_task_item(product_queue):
        try:
            item = product_queue.get(**get_kwargs)

        except queue.Empty:
            item = None

        return item

    return get_task_item

In [ ]:
#| exporti

def workloop(
        task_queue: queue.Queue,
        worker_fn:  callable,
        product_queue: queue.Queue,
        stop_event: threading.Event,
    ):
        """
        A worker thread loop that gets items from a queue, processes them with a worker function and puts it's result on a queue.

        Each item is a tuple of (args, kwargs) to be passed to the worker function.
    
        """

        sleepfor = 0.05 # initial sleep time when no work is available
        sleepmax = 0.5  # max sleep time when no work is available

        # indefinately keep getting new items from the queue to process
        while True:
            try:
                item = ()
                try:
                    # by default queue.get() waits for new items
                    item = task_queue.get_nowait()
                    task_queue.task_done()
                    sleepfor = 0.05

                except queue.Empty:
                    if stop_event.is_set():
                        break
                    else:
                        time.sleep(sleepfor)
                        sleepfor = min(sleepfor * 1.2, sleepmax)

                if item:
                    try:
                        result = worker_fn(*item[0], **item[1])
                        if result is not None:
                            product_queue.put_nowait(result)

                    except queue.Full:
                        pass


            except Exception as x:
                syslog.exception("Exception: %s", x, exc_info=True, stack_info=True)



In [ ]:
import nbdev; nbdev.nbdev_export()